Run the Shopflow dbt project every morning with Airflow.

In [5]:
from datetime import datetime
from pathlib import Path

In [2]:
from airflow import DAG
from airflow.providers.standard.operators.bash import BashOperator

In [11]:
execute_datetime = datetime(2026, 11, 1)

In [12]:
PROJECT = Path.cwd().resolve().parent
DBT = f"cd {PROJECT} && {PROJECT}/.venv/bin/dbt"
PY = f"cd {PROJECT} && {PROJECT}/.venv/bin/dbt"
PAPERMILL = f"papermill load_raw.ipynb output_load_raw_{datetime.strftime(execute_datetime, '%Y-%m-%d')}.ipynb"

In [13]:
print(PROJECT, DBT, PY, PAPERMILL)

D:\Learn dbt cd D:\Learn dbt && D:\Learn dbt/.venv/bin/dbt cd D:\Learn dbt && D:\Learn dbt/.venv/bin/dbt papermill load_raw.ipynb output_load_raw_2026-11-01.ipynb


In [ ]:
with DAG(
    dag_id="shopflow_dbt",
    start_date=datetime(2026, 11, 1),
    schedule="0 6 * * *",
    catchup=False,
    tags=["dbt", "shopflow"]
) as dag:
    load = BashOperator(task_id="load_raw", bash_command=f"{PAPERMILL}")
    fresh = BashOperator(task_id="source_freshness", bash_command=f"{DBT} source freshness")
    build = BashOperator(task_id="dbt_build", bash_command=f"{DBT} build --selector nightly")
    docs = BashOperator(task_id="dbt_docs", bash_command=f"{DBT} docs generate")
    
    load >> fresh >> build >> docs

In [ ]:
# Run the env set first and then run this module
# > soure airflow_env.sh && airflow db migrate 2>&1 | tail
# > ./airflow_env.bat; airflow db migrate
# > source airflow_env.sh && airflow dags reserialize > /dev/null 2>&1 && airflow dags list 2>/dev/null | grep -E 'dag_id|shopflow'


In [14]:
# Save the log at a particular stage of run
# > source airflow_env.sh ** airflow dags test shopflow_dbt 2026-11-09 > logs/airflow_test.log 2>&1; grep -oE 'end task task_id=[a-z_]+|DagRun Finished: dag_id=[a-z_]+|state=[a-z_]+, run_type' logs/airflow_test.log